# PTQ (Dynamic Range)

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load Baseline Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


## LiteRT 모델로 변환 (PTQ (DR))

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] # default optimization strategy (enable post-training quantization)
tflite_model = converter.convert()

In [ ]:
tflite_ptq_dr_file = save_dir + 'mnist_ptq_dr.tflite'
open(tflite_ptq_dr_file, 'wb').write(tflite_model)

64200

## File size 비교 : Baseline model vs PQT (DR) model

In [ ]:
import os
tflite_baseline_model_file = save_dir + 'mnist_baseline_model.tflite'
print("Size of Baseline LiteRT Model file : {}".format(os.path.getsize(tflite_baseline_model_file)))
print("Size of PTQ(DR) LiteRT Model file : {}".format(os.path.getsize(tflite_ptq_dr_file)))

Size of Baseline LiteRT Model file : 233596
Size of PTQ(DR) LiteRT Model file : 64200


## LiteRT 설치 및 Interpreter 로딩

In [ ]:
!pip install ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 82.0 MB/s eta 0:00:00


In [ ]:
from ai_edge_litert.interpreter import Interpreter

## interpreter 생성 (Baseline model)

In [ ]:
interpreter_base = Interpreter(model_path=str(tflite_baseline_model_file))
interpreter_base.allocate_tensors()

In [ ]:
input_details = interpreter_base.get_input_details()
output_details = interpreter_base.get_output_details()

In [ ]:
input_details

[{'name': 'serving_default_input_1:0',
  'index': 0,
  'shape': array([ 1, 28, 28,  1], dtype=int32),
  'shape_signature': array([-1, 28, 28,  1], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0,
   'block_size': 0},
  'sparsity_parameters': {}}]

In [ ]:
output_details

[{'name': 'StatefulPartitionedCall:0',
  'index': 17,
  'shape': array([ 1, 10], dtype=int32),
  'shape_signature': array([-1, 10], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0,
   'block_size': 0},
  'sparsity_parameters': {}}]

## Check Dtype and shape of Input/Output Tensor

In [ ]:
input_dtype = input_details[0]['dtype']
output_dtype = output_details[0]['dtype']

input_shape = input_details[0]['shape']
output_shape = output_details[0]['shape']

print("input_dtype : {}".format(input_dtype))
print("output_dtype : {}".format(output_dtype))

print("input_shape : {}".format(input_shape))
print("output_shape : {}".format(output_shape))

input_dtype : <class 'numpy.float32'>
output_dtype : <class 'numpy.float32'>
input_shape : [ 1 28 28  1]
output_shape : [ 1 10]


## interpreter 생성 (PTQ (DR))

In [ ]:
interpreter_ptq_dr = Interpreter(model_path=str(tflite_ptq_dr_file))
interpreter_ptq_dr.allocate_tensors()

## input detail / output detail 확인 (PTQ (DR))

In [ ]:
input_details = interpreter_ptq_dr.get_input_details()
output_details = interpreter_ptq_dr.get_output_details()

input_dtype = input_details[0]['dtype']
output_dtype = output_details[0]['dtype']

input_shape = input_details[0]['shape']
output_shape = output_details[0]['shape']

print("input_dtype : {}".format(input_dtype))
print("output_dtype : {}".format(output_dtype))

print("input_shape : {}".format(input_shape))
print("output_shape : {}".format(output_shape))

input_dtype : <class 'numpy.float32'>
output_dtype : <class 'numpy.float32'>
input_shape : [ 1 28 28  1]
output_shape : [ 1 10]


## 추론 실행 (PTQ (DR))

In [ ]:
test_image = np.expand_dims(test_images[0], axis=0)

input_index = interpreter_ptq_dr.get_input_details()[0]["index"]
output_index = interpreter_ptq_dr.get_output_details()[0]["index"]

interpreter_ptq_dr.set_tensor(input_index, test_image)
interpreter_ptq_dr.invoke()
predictions = interpreter_ptq_dr.get_tensor(output_index)

print(predictions)
print(np.argmax(predictions))
print(test_labels[0])

[[2.7281031e-12 3.2476892e-12 8.6881738e-11 1.4535459e-08 1.1000996e-17
  2.2470547e-14 3.8090406e-19 1.0000000e+00 1.9152763e-12 1.9089279e-11]]
7
7


## Test data 기반 accuracy 평가

In [ ]:
def evaluate_model(interpreter):
  input_index = interpreter.get_input_details()[0]["index"]
  output_index = interpreter.get_output_details()[0]["index"]

  prediction_digits = []
  for test_image in test_images:
    test_image = np.expand_dims(test_image, axis=0).astype(np.float32)
    interpreter.set_tensor(input_index, test_image)

    interpreter.invoke()

    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  accurate_count = 0
  for index in range(len(prediction_digits)):
    if prediction_digits[index] == test_labels[index]:
      accurate_count += 1
  accuracy = accurate_count * 1.0 / len(prediction_digits)

  return accuracy

In [ ]:
print(evaluate_model(interpreter_base))
print(evaluate_model(interpreter_ptq_dr))

0.9904
0.9902


## 추론 속도 측정

* benchmark_model 설치

In [ ]:
!wget https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
!chmod +x linux_x86-64_benchmark_model

--2025-12-21 14:47:24--  https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.203.207, 74.125.199.207, 142.251.188.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.203.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6685264 (6.4M) [application/octet-stream]
Saving to: ‘linux_x86-64_benchmark_model’

linux_x86-64_benchm 100%[===================>]   6.38M  --.-KB/s    in 0.03s   

2025-12-21 14:47:24 (224 MB/s) - ‘linux_x86-64_benchmark_model’ saved [6685264/6685264]



* Baseline model

In [ ]:
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_baseline_model_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_baseline_model.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.233596
INFO: Initialized session in 5.113ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=12307 first=102 curr=38 min=33 max=641 avg=40.3824 std=9 p5=37 median=38 p95=54

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=22820 first=57 curr=40 min=33 max=687 avg=43.5648 std=13 p5=37 median=40 p95=72

INFO: Inference timings in us: Init: 5113, First inference: 102, Warmup (avg): 40.3824, Inference (a

* PTQ (DR)

In [ ]:
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_ptq_dr_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_ptq_dr.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_ptq_dr.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_ptq_dr.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.0642
INFO: Initialized session in 3.898ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=11116 first=82 curr=42 min=35 max=586 avg=44.729 std=12 p5=39 median=39 p95=71

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=24082 first=58 curr=68 min=35 max=202 avg=41.2985 std=5 p5=39 median=39 p95=53

INFO: Inference timings in us: Init: 3898, First inference: 82, Warmup (avg): 44.729, Inference (avg): 41.2985
INFO: Note: as th